# AWI Pipeline , Awareness Index (Unified)

Computes the **Awareness Index (AWI)** and related enrichment metrics for every player × phase across all 5 Bundesliga matches in a single S3 read per phase.

**Metrics computed per player × phase:**
- `awi_per_minute` , full-phase scan rate (scans/min)
- `coverage_pct` , fraction of phase frames where the player's skeleton was tracked
- `hbd_mean_deg` , head-body decoupling angle (blind-side scan proxy)
- `pre_pass_awi` , scan rate in the 5 s window before each pass
- `scan_blindside_pct` , fraction of scans where head > 120° from body
- `scan_forward_pct` / `scan_lateral_pct` , directional scan ratios

**Outputs:**
- `results/awi_full.csv` , 400 rows (20 players × 2 halves × 5 matches × 2 teams)
- `results/match_moments.csv` , top-5 highest pre-pass scan moments per match

**Checkpoint:** completed phases are saved after each phase and skipped on re-run.  
After AWS token expiry: `aws sso login --profile <profile>`, re-run Cell 0, re-run Cell 2.

## Step 1. Imports and AWS Session Setup

Initialises the AWS session and S3 client, and sets the S3 bucket and challenge prefix constants.

In [ ]:
# Cell 1: Imports and AWS session setup

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import os
import time
import importlib
import pandas as pd

from src.eda_helpers import (
    create_session,
    list_bucket,
    load_json,
    load_xml,
)

SESSION_PROFILE = os.environ.get("AWS_PROFILE")
REGION = os.environ.get("AWS_DEFAULT_REGION", "eu-central-1")
BUCKET = os.environ.get("HACKATHON_BUCKET", "your-s3-bucket-name")
CHALLENGE_PREFIX = "Challenge 2 – Unlock the Power of 3D Football Data/Match_Data"

session, s3_client, s3fs = create_session(SESSION_PROFILE, REGION)
if session is not None:
    print(f"[OK] AWS session created (profile={SESSION_PROFILE}, region={REGION}, bucket={BUCKET})")
else:
    print(f"[ERROR] Failed to create AWS session , run: aws sso login --profile {SESSION_PROFILE}")

ModuleNotFoundError: No module named 'src'

## Unified Pipeline , AWI + Enrichment in One Pass

Runs **all KPIs in a single S3 read per phase** , no second pass for enrichment.

Each phase is read once and produces:
- **AWI** , scans/min, coverage_pct
- **HBD** , head-body decoupling (blind-side scan proxy)
- **Pre-pass AWI** , scan rate in the 5s before each pass
- **Scan direction** , forward / lateral / blind-side ratios

Output: `results/awi_full.csv` , one wide row per player × phase with all metrics.  
Match moments (top-5 highest pre-pass scan moments per match): `results/match_moments.csv`

**Checkpoint**: completed phases are saved after each phase and skipped on re-run.  
After AWS token expiry: `aws sso login --profile <profile>`, re-run Cell 0, re-run Cell 2.


## Step 2. Verify Events XML Paths

Checks that the Events XML file for each match is accessible in S3 before starting the pipeline.  
These files are used to extract pass timestamps for the pre-pass AWI calculation.

In [ ]:
# Cell 2: Define Events XML keys and verify S3 paths

EVENTS_XML_KEYS = {
    "FCB-HSV": "Bayern_Hamburg/Events_Bayern_Hamburg.xml",
    "BVB-VFB": "Dortmund_Stuttgart/Events_Dortmund_Stuttgart.xml",
    "SGE-FCB": "Frankfurt_Bayern/Events_Frankfurt_Bayern.xml",
    "SGE-FCU": "Frankfurt_Union/Events_Frankfurt_Union.xml",
    "FCU-FCB": "Union_Bayern/Events_Union_Bayern.xml",
}

print("Verifying Events XML paths...")
for match_id, rel_key in EVENTS_XML_KEYS.items():
    full_key = f"{CHALLENGE_PREFIX}/{rel_key}"
    try:
        s3_client.head_object(Bucket=BUCKET, Key=full_key)
        print(f"  OK      {match_id}: {rel_key}")
    except Exception:
        print(f"  MISSING {match_id}: {full_key}")


Verifying Events XML paths...
  OK      FCB-HSV: Bayern_Hamburg/Events_Bayern_Hamburg.xml
  OK      BVB-VFB: Dortmund_Stuttgart/Events_Dortmund_Stuttgart.xml
  OK      SGE-FCB: Frankfurt_Bayern/Events_Frankfurt_Bayern.xml
  OK      SGE-FCU: Frankfurt_Union/Events_Frankfurt_Union.xml
  OK      FCU-FCB: Union_Bayern/Events_Union_Bayern.xml


## Step 3. Run Unified Pipeline

Runs **all KPIs in a single S3 read per phase** , no second pass for enrichment.

Each phase is read once and produces:
- **AWI** , scans/min, coverage_pct
- **HBD** , head-body decoupling (blind-side scan proxy)
- **Pre-pass AWI** , scan rate in the 5s before each pass
- **Scan direction** , forward / lateral / blind-side ratios

Output: `results/awi_full.csv` , one wide row per player × phase with all metrics.  
Match moments (top-5 highest pre-pass scan moments per match): `results/match_moments.csv`

**Checkpoint**: completed phases are saved after each phase and skipped on re-run.  
After AWS token expiry: `aws sso login --profile <profile>`, re-run Cell 0, re-run Cell 2.


In [ ]:
# Cell 3: Unified pipeline , all 5 matches, all KPIs, one S3 read per phase
#
# Computes per player × phase in a single vectorized pass:
#   awi_per_minute      , full-phase scan rate
#   coverage_pct        , skeleton data quality
#   hbd_mean_deg        , head-body decoupling (blind-side proxy)
#   pre_pass_awi        , scan rate in 5s before each pass
#   scan_blindside_pct  , fraction of scans where head > 120° from body
#   scan_forward_pct    , fraction of forward-facing scans
#   scan_lateral_pct    , fraction of lateral scans
#
# Checkpoint: results/awi_full.csv , completed phases skipped on re-run.
# EVENTS_XML_KEYS must be defined (run Cell 1 first).

import src.batch_pipeline as bp
import src.skeleton_parser as sp
import src.awi_calculator as awi
import src.event_parser as ep
import src.pre_pass_awi as ppa
for mod in (bp, sp, awi, ep, ppa):
    importlib.reload(mod)

from src.batch_pipeline import run_all_matches_unified, MATCH_CONFIGS

os.makedirs("results", exist_ok=True)
CHECKPOINT = "results/awi_full.csv"
MOMENTS    = "results/match_moments.csv"

print(f"Unified pipeline: {len(MATCH_CONFIGS)} matches, all KPIs in one S3 read per phase")
t0 = time.time()

all_df = run_all_matches_unified(
    s3_client, s3fs, BUCKET, CHALLENGE_PREFIX,
    match_configs=MATCH_CONFIGS,
    events_xml_keys=EVENTS_XML_KEYS,
    checkpoint_path=CHECKPOINT,
    moments_path=MOMENTS,
)

elapsed = time.time() - t0
print(f"\nTotal time  : {elapsed/60:.1f} min")
print(f"Total rows  : {len(all_df):,}")
print(f"Saved to    : {CHECKPOINT}")
print()
print(all_df.groupby("match_id").size().rename("rows").to_string())
print()
enriched = all_df[all_df["pre_pass_awi"].notna()]
print(f"Rows with pre-pass AWI: {len(enriched)}")
if not enriched.empty:
    print(enriched[["awi_per_minute", "pre_pass_awi", "hbd_mean_deg",
                     "scan_blindside_pct"]].describe().round(2).to_string())
print()
if os.path.exists(MOMENTS):
    moments_df = pd.read_csv(MOMENTS)
    print(f"Match moments saved: {len(moments_df)} rows")
    print(moments_df.sort_values('pre_pass_scan_count', ascending=False)
          [["match_id", "name", "minute", "pre_pass_scan_count", "phase_label"]]
          .head(10).to_string(index=False))


Unified pipeline: 5 matches, all KPIs in one S3 read per phase

=== FCB-HSV: FC Bayern Muenchen vs Hamburger SV ===
  [FCB-HSV] 40 players, 2 phases
  [FCB-HSV] Events XML loaded: 1084 passes
  [FCB-HSV] 1st half: frames 3,330,943 – 3,484,329 ... 103s
  [FCB-HSV] 2nd half: frames 3,536,417 – 3,678,119 ... 103s
  [FCB-HSV] Top-5 moments saved to results/match_moments.csv
  Done: 80 player-phase rows

=== BVB-VFB: Borussia Dortmund vs VfB Stuttgart ===
  [BVB-VFB] 40 players, 2 phases
  [BVB-VFB] Events XML loaded: 898 passes
  [BVB-VFB] 1st half: frames 2,790,134 – 2,935,531 ... 86s
  [BVB-VFB] 2nd half: frames 2,984,098 – 3,141,386 ... 85s
  [BVB-VFB] Top-5 moments saved to results/match_moments.csv
  Done: 80 player-phase rows

=== SGE-FCB: Eintracht Frankfurt vs FC Bayern Muenchen ===
  [SGE-FCB] 40 players, 2 phases
  [SGE-FCB] Events XML loaded: 1138 passes
  [SGE-FCB] 1st half: frames 3,331,222 – 3,476,767 ... 88s
  [SGE-FCB] 2nd half: frames 3,526,374 – 3,674,654 ... 75s
  [SGE-F